## <center><u>This notebook is for the <span style="color:red">functions</span> from the earlier <span style="color:blue">Gemini</span> optimizer notebook to be imported into our <span style="color:green">Flask</span> UI application.</u></center>

In [1]:
# working with our os and .env file
import os
from dotenv import load_dotenv

# since we're using Gemini
import google.generativeai as genai

# make printouts look nicer
from IPython.display import display, Markdown
from markdown import markdown

In [2]:
load_dotenv()

True

#### <u><span style="color:blue">Configuring the Gemini API</span></u>
> ##### Running 'load_dotenv()' loads the environment variables. If the API key is in there, it will load and the API call will be made.
> ##### If the key was not loaded, the function will tell you there was a problem.
***

In [3]:
def configure_gemini_api():
    """
    This function does two things:
    1.) It configures the Gemini API using the 'GEMINI_API_KEY' environment variable; and
    2.) Let's you know if that environment variable was not loaded.
    """

    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY was not loaded from your environment.")
    genai.configure(api_key=GEMINI_API_KEY)

    # No 'return' statement is needed - all this does is perform an action, not any calculation

#### <u><span style="color:blue">Structure Your Response by Defining Your Function Schema</u></span>
> ##### If you don't tell <span style="color:blue">Gemini</span> the structure you want, you won't fully harness its capabilities.

In [4]:
def tailored_resume_function_schema():
    """
    Dictates to Gemini how you want your resume tailored based on a job description.
    What returns is a dictionary that allows us to pass everything to the Gemini API in its expected format.
    """
    function_schema = {
        "name": "tailor_resume",
        "description": "Optimize my resume based on a job description.",
    
        # 'Here is how we want the info back'
        "parameters": {
            "type": "object",
            "properties": {
                "tailored_resume": {
                    "type": "string",
                    "description": "The Markdown-formatted resume.",
                },
                "additional_suggestions": {
                    "type": "string",
                    "description": "Ideas to make your resume stronger for this position.",
                },
            },
    
            # Make sure we get back SOMETHING
            "required": ["tailored_resume"],
        },
    }
    return function_schema

#### <u><span style="color:blue">Instantiate the Gemini Model</u></span>
> ##### Tell the model to use the <span style="color:blue">gemini-2.0-flash</span> engine.
> ##### It's important to instantiate the model because you're essentially making it an actual, defined "thing" the program can work with.

In [5]:
def gemini_initialization_with_function_calling(function_schema: dict):
    """
    By passing in the (expected) dictionary returned from 'function_schema,' you make the gemini-2.0-flash engine
    a single variable ("initialized") that can be used with all its parameters - all its rules. 
    
    Returns the initialized GenerativeModel with its function-calling "tools"
    """
    model = genai.GenerativeModel(
        model_name = "gemini-2.0-flash",

        # Pass the function_schema for the model to call
        tools = [{"function_declarations" : [function_schema]}]
    )
    return model

#### <u><span style="color:blue">Prompt Function</u></span>
> ##### Debated functionalizing this because prompts can always be improved upon.
> ##### However, I decided to go ahead and do it (with hints) to keep the <span style="color:green">Flask</span> notebook clean.

In [6]:
def generate_tailoring_prompt(resume_text: str, jd_string: str) -> str:
    """
    Generates the prompt telling Gemini to tailor a resume based on a given job description.

    Arguments are the 'resume_text' and 'jd_string' variables.

    What gets returned is the prompt from the 'res_optimizer_new_and_improved_gemini' notebook.
    """
    return f"""
    You are a professional resume optimization expert.
    Optimize the resume I have provided you to align with the given job description following the guidelines below:

    1. Make the resume one page, relevant, keyword optimized, action-driven.
    2. Format it cleanly in **Markdown**.
    3. The primary goal of this optimization is to best position the resume against Application Tracking Systems.
    4. At the end, provide an "**Additional Suggestions**" section with suggestions improvements my resume where gaps exist.
    

    Resume:
    {resume_string}

    Job Description:
    {jd_string}
    """

#### <u><span style="color:blue">Send Your Prompt To the gemini-2.0-flash Model</u></span>
> ##### Send it along with the function schema so that <span style="color:blue">Gemini</span> knows to call on it
> ##### There are a lot of arguments along with this function, each broken down with hash notes.

#### <center><b><u><span style="color:red">Please Note</span></u>:</b></center>
##### The following code block has been commented out but left in to show the resolution process, not only for this notebook, but also for what appears to be a pretty common problem when working with <span style="color:blue">genai</span>'s <span style="color:blue">GenerateContentResponse</span> attribute.

In [7]:
# def get_gemini_response_with_function_calling(
#     # the model initialized in the earlier 'gemini_initialization_with_function_calling' function
#     model: genai.GenerativeModel,
#     # the prompt string returned from 'generate_tailoring_prompt'
#     prompt: str,
#     # the dictionary returned from 'tailored_resume_function_schema'
#     function_schema: dict,
#     # output randomness and tone - using 0.7 here as with ChatGPT
#     temperature: 0.7
#     # designate the return data type I'm looking for (the '->') as 'GenerateContentResponse' from the 'google.generativeai' library
#     # and yes, this return type designation was helped along by Gemini itself.
# ) -> genai.GenerateContentResponse:
#     """
#     This function sends our prompt to Gemini along with the function schema.
#     It returns the response object from the gemini-2.0-flash engine - the 'genai.GenerateContentResponse' line
#     """
#     response = model.generate_content(
#         contents=[{"role": "user", "parts": [{"text": prompt}]}],
#         tools=[{"function_declarations": [function_schema]}],
#         generation_config=genai.types.GenerationConfig(temperature=temperature),
#     )
#     return response

***
### <center><i><span style="color:red">^^ Welp! We hit a sticking point... ^^</span></i></center>
***
#### Looks like '<span style="color:blue">GenerateContentResponse</span>' is not a recognizable attribute in '<span style="color:blue">google.generativeai.</span>'
##### <u><b><span style="color:deeppink">To remedy this, we're doing the following</u></span>:</b>
##### 1.) Inspecting the '<span style="color:blue">genai,/span>' module to see all its attributes and submodules;
##### 2.) Taking that list of names and looking for '<span style="color:blue">GenerateContentResponse</span>' or something that might contain it;
##### 3.) Inspecting those potential submodules; and either:
> ##### a.) <span style="color:firebrick">Updating the library</span>; or 
> ##### b.) <span style="color:firebrick">Modifying the import and type hint in the function's title</span>.

In [8]:
# Inspecting genai module
dir(genai)

['ChatSession',
 'GenerationConfig',
 'GenerativeModel',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'annotations',
 'caching',
 'configure',
 'create_tuned_model',
 'delete_file',
 'delete_tuned_model',
 'embed_content',
 'embed_content_async',
 'get_base_model',
 'get_file',
 'get_model',
 'get_operation',
 'get_tuned_model',
 'list_files',
 'list_models',
 'list_operations',
 'list_tuned_models',
 'protos',
 'responder',
 'string_utils',
 'types',
 'update_tuned_model',
 'upload_file',
 'utils']

#### Doing some homework into these, and it appears '<span style="color:blue">GenerateContentResponse</span>' is most likely in the sub-module '<span style="color:blue">types</span>.'

In [9]:
dir(genai.types)

['Any',
 'AnyModelNameOptions',
 'AsyncGenerateContentResponse',
 'BaseModelNameOptions',
 'BlobDict',
 'BlobType',
 'BlockedPromptException',
 'BlockedReason',
 'BrokenResponseError',
 'CallableFunctionDeclaration',
 'CitationMetadataDict',
 'CitationSourceDict',
 'ContentDict',
 'ContentFilterDict',
 'ContentType',
 'ContentsType',
 'File',
 'FileDataDict',
 'FileDataType',
 'FunctionDeclaration',
 'FunctionDeclarationType',
 'FunctionLibrary',
 'FunctionLibraryType',
 'GenerateContentResponse',
 'GenerationConfig',
 'GenerationConfigDict',
 'GenerationConfigType',
 'HarmBlockThreshold',
 'HarmCategory',
 'HarmProbability',
 'IncompleteIterationError',
 'Model',
 'ModelNameOptions',
 'ModelsIterable',
 'PartDict',
 'PartType',
 'Permission',
 'Permissions',
 'RequestOptions',
 'RequestOptionsType',
 'SafetyFeedbackDict',
 'SafetyRatingDict',
 'SafetySettingDict',
 'Status',
 'StopCandidateException',
 'StrictContentType',
 'Tool',
 'ToolDict',
 'ToolsType',
 'TunedModel',
 'TunedMode

#### And indeed it is. This means it was installed properly and we don't need to pip update anything. What we can then do is Option B from above: modify the type hint in the '<span style="color:dodgerblue">get_gemini_response_with_function_calling</span>' function title: 

In [10]:
# Rewriting 'get_gemini_response_with_function_calling' with same notes:
def get_gemini_response_with_function_calling(
    model: genai.GenerativeModel,
    prompt: str,
    function_schema: dict,
    temperature: 0.7
    # modified hint:
) -> genai.types.GenerateContentResponse:
    """
    This function sends our prompt to Gemini along with the function schema.
    It returns the response object from the gemini-2.0-flash engine - the 'genai.GenerateContentResponse' line
    """
    response = model.generate_content(
        contents=[{"role": "user", "parts": [{"text": prompt}]}],
        tools=[{"function_declarations": [function_schema]}],
        generation_config=genai.types.GenerationConfig(temperature=temperature),
    )
    return response

#### <center><span style="color:green">Success!!!</span> Now, moving on...</center>

***
#### <u><span style="color:blue">Handle the Gemini Response</span></u>
##### <u>This is a pretty loaded function, so a quick summation of what's going on:</u>
> ##### a.) The '<span style="color:firebrick">if response.candidates</span>' block checks to see if <span style="color: blue">Gemini</span> gave us any responses (aka: "<span style="color: blue">candidates</span>"), what their content was, how they are broken up. It also checks for function calls and executes accordingly;
> ##### b.) it then generates the tailored resume and offers suggestions to (<b>hopefully</b>) improve the candidate's odds at getting past the company's applicant tracking system (ATS); and
> ##### c.) safeguards / exception-error codes make up the tail-end of the function to direct the user on where prgram errors / corrections can be made.
***

In [11]:
def handle_gemini_response(response: genai.types.GenerateContentResponse, output_file_path: str = "nj_gemini_pm_res.md"):
    """
    This function is designed to handle the Gemini API response. It checks for either blob text orfunction calls, 
    and writes the file 'nj_gemini_pm_res.md.'

    Explanation of arguments:
        response (genai.types.GenerateContentResponse) == the specified response object from the Gemini model.
        output_file_path (str, optional) == where the tailored resume is written (defaults to "nj_gemini_pm_res.md").
    """
    if response.candidates:
        first_candidate = response.candidates[0]

        if first_candidate.content and first_candidate.content.parts:
            first_part = first_candidate.content.parts[0]

            if first_part.function_call:
                function_call = first_part.function_call

                if function_call.name == "tailor_resume":
                    arguments = function_call.args
                    tailored_resume = arguments.get("tailored_resume")
                    additional_suggestions = arguments.get("additional_suggestions")

                    if tailored_resume:
                        print("Tailored Resume:")
                        print(tailored_resume)

                        try:
                            with open(output_file_path, "w", encoding="utf-8") as output_file:
                                output_file.write(tailored_resume)
                            print(f"Saved Gemini-tailored resume to {output_file_path}")
                        except Exception as e:
                            print(f"There's been a problem: we couldn't save the Markdown file: {e}")

                    else:
                        print("Error: tailored_resume not found in function arguments.")

                    if additional_suggestions:
                        print("\nAdditional Suggestions:")
                        print(additional_suggestions)
                else:
                    print("Function call was made, but not for 'tailor_resume'.")
                    print("Function name:", function_call.name)

            elif first_part.text:
                print("Gemini's direct response:")
                print(first_part.text)
            else:
                print("No function call or text in the response.")
                print(response)
        else:
            print("Response contained no content or parts")
            print(response)
    else:
        print("No candidates in response")
        print(response)

#### ^^^ All functions have been moved to a separate Python file (<b><span style="color:green">gemini_flask_functions.py</b></span>) for Import into the <b><span style="color:blue">Fla</span><span style="color:gold">sk</span> UI</b> Notebook.